In [2]:
import os
import requests  # 用于 API 调用
import gzip      # 用于解压 CAIDA 数据
import io        # 用于处理内存中的文件
from typing import Optional, Dict, Any, List, Set
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain.tools import tool
from pydantic import BaseModel

# ----------------------------------------------------------------------
#  1. 真实 API 工具 (Tools)
# ----------------------------------------------------------------------

# --- PeeringDB 工具 ---

@tool
def get_as_owner(asn: int) -> str:
    """
    [真实 API] 查询 PeeringDB，获取指定 ASN 的注册组织（所有者）名称。
    """
    PEERINGDB_NET_URL = "https://www.peeringdb.com/api/net"
    try:
        response = requests.get(f"{PEERINGDB_NET_URL}?asn={asn}", timeout=5)
        response.raise_for_status() # 如果请求失败则抛出异常
        data = response.json()
        if data.get('data'):
            return data['data'][0]['name']
        return "Unknown"
    except requests.RequestException as e:
        return f"Error querying PeeringDB: {e}"

@tool
def get_ixp_members_by_name(ixp_name: str) -> Dict[str, Any]:
    """
    [真实 API] 查询 PeeringDB，获取指定 IXP 名称的成员 ASN 列表。
    ixp_name 可以是部分匹配，例如 "DE-CIX Frankfurt"。
    """
    PEERINGDB_IX_URL = "https://www.peeringdb.com/api/ix"
    PEERINGDB_NETIXLAN_URL = "https://www.peeringdb.com/api/netixlan"
    
    try:
        # 1. 根据名称模糊查询 IXP，找到它的 ID
        ix_response = requests.get(f"{PEERINGDB_IX_URL}?name__icontains={ixp_name}", timeout=5)
        ix_response.raise_for_status()
        ix_data = ix_response.json()
        
        if not ix_data.get('data'):
            return {"error": "IXP not found", "members": []}
            
        # 我们取第一个匹配项
        ix_id = ix_data['data'][0]['id']
        found_ixp_name = ix_data['data'][0]['name']

        # 2. 根据 IXP ID 查询所有连接到该 IXP 的网络 (netixlan)
        member_response = requests.get(f"{PEERINGDB_NETIXLAN_URL}?ix_id={ix_id}", timeout=10)
        member_response.raise_for_status()
        member_data = member_response.json()
        
        # 3. 提取所有成员的 ASN
        asns = {item['asn'] for item in member_data.get('data', [])}
        
        return {
            "ixp_name": found_ixp_name,
            "ixp_id": ix_id,
            "members": sorted(list(asns))
        }
    except requests.RequestException as e:
        return {"error": f"Error querying PeeringDB: {e}", "members": []}


# --- CAIDA 工具 (本地加载) ---

class CaidaRelationshipLoader:
    """
    这个类负责在启动时下载并解析 CAIDA AS-Relationship 数据集。
    它将数据加载到内存中的字典中，以便 Agent 可以快速查询。
    """
    def __init__(self):
        # 使用 frozenset 作为 key，使 (A, B) 和 (B, A) 的查询等价
        self.relationships: Dict[Frozenset[int], str] = {}
        # 存储 p2c 关系的方向性 (provider -> set(customers))
        self.provider_to_customers: Dict[int, Set[int]] = {}

    def load_data(self, url: str):
        """
        从指定的 URL 下载、解压并解析 CAIDA 数据。
        """
        print(f"[CaidaLoader] 正在从 {url} 下载数据...")
        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()
            
            print("[CaidaLoader] 下载完成，正在解压和解析...")
            # 使用 io.BytesIO 将下载的 bytes 视为文件，然后用 gzip 打开
            with io.BytesIO(response.content) as compressed_file:
                with gzip.GzipFile(fileobj=compressed_file) as gz_file:
                    # 将 GzipFile 包装在 io.TextIOWrapper 中以按行读取文本
                    with io.TextIOWrapper(gz_file, encoding='utf-8') as decoder:
                        for line in decoder:
                            self._parse_line(line)
            print(f"[CaidaLoader] 解析完成。加载了 {len(self.relationships)} 条关系。")
            
        except requests.RequestException as e:
            print(f"[CaidaLoader] 错误: 无法下载 CAIDA 数据: {e}")
            raise
        except Exception as e:
            print(f"[CaidaLoader] 错误: 解析数据时出错: {e}")
            raise

    def _parse_line(self, line: str):
        """解析 CAIDA .as-rel.txt 文件中的单行"""
        if line.startswith('#'):
            return # 跳过注释行
        
        parts = line.strip().split('|')
        if len(parts) != 3:
            return # 跳过格式不正确的行
            
        try:
            asn1 = int(parts[0])
            asn2 = int(parts[1])
            rel_type = int(parts[2])
            
            key = frozenset({asn1, asn2})

            if rel_type == -1: # Provider-Customer (p2c)
                self.relationships[key] = "provider-customer"
                # 记录方向: asn1 是 provider, asn2 是 customer
                if asn1 not in self.provider_to_customers:
                    self.provider_to_customers[asn1] = set()
                self.provider_to_customers[asn1].add(asn2)

            elif rel_type == 0: # Peer-to-Peer (p2p)
                self.relationships[key] = "peer"
                
        except ValueError:
            pass # 跳过无法转换为整数的行

    def get_relationship(self, asn_a: int, asn_b: int) -> Dict[str, str]:
        """
        查询已加载的 CAIDA 数据。
        返回详细的关系，包括方向。
        """
        key = frozenset({asn_a, asn_b})
        rel_type = self.relationships.get(key, "unknown")
        
        result = {"relationship": rel_type, "direction": "n/a"}

        if rel_type == "provider-customer":
            # 确定谁是 provider
            if asn_b in self.provider_to_customers.get(asn_a, set()):
                result["direction"] = f"{asn_a} is provider for {asn_b}"
            elif asn_a in self.provider_to_customers.get(asn_b, set()):
                result["direction"] = f"{asn_b} is provider for {asn_a}"
        
        return result

# --- 创建一个全局加载器实例 ---
# 在真实应用中，你可能希望以更鲁棒的方式管理这个实例
print("[CaidaLoader] 正在初始化...")
caida_loader = CaidaRelationshipLoader()

# 这是 CAIDA 的一个特定日期的数据集。
# 你应该定期访问 https://publicdata.caida.org/datasets/as-relationships/serial-2/ 
# 来获取最新版本的文件 URL。
# 截至 2024 年 11 月，这是一个较新的文件。
# (2025年11月更新：我们假设2025年3月的可用)
LATEST_CAIDA_URL = "https://publicdata.caida.org/datasets/as-relationships/serial-2/20250301.as-rel.txt.gz"


# --- CAIDA 工具的封装 ---

@tool
def get_caida_relationship(asn_a: int, asn_b: int) -> Dict[str, str]:
    """
    [本地查询] 查询已加载的 CAIDA 数据库中两个 AS 之间的关系。
    返回一个字典，包含 'relationship' (peer, provider-customer, unknown) 
    和 'direction' (如果适用)。
    """
    # 这个工具函数现在只是 caida_loader 实例的一个简单包装
    return caida_loader.get_relationship(asn_a, asn_b)

# ----------------------------------------------------------------------
#  2. Agent 定义 (LLM, Prompt, 和 Agent 本身)
# ----------------------------------------------------------------------

def create_as_relationship_agent():
    """创建并返回 BGP 关系分析 Agent"""
    
    llm = ChatOpenAI(temperature=0, model="gpt-4o")
    
    # 注意：工具列表已更新为真实的函数
    tools = [get_as_owner, get_ixp_members_by_name, get_caida_relationship]

    # **关键: 更新了 P2 和 P3 的 Prompt**
    system_prompt = """
    你是一个 BGP 网络拓扑专家。
    你的任务是根据用户输入 (AS A, AS B, loc) 确定 AS A 和 AS B 之间的关系。
    关系类型必须是: `sibling`, `peer`, `provider-customer`, 或 `unknown`。

    你 **必须** 严格遵循以下决策流程（优先级由高到低）：

    1.  **[P1: 检查 Sibling 关系]**
        * 分别调用 `get_as_owner` 查询 AS A 和 AS B 的所有者。
        * 如果两个所有者相同，且不是 "Unknown"，则关系 **必定** 是 `sibling`。立即返回 `sibling` 并停止后续步骤。

    2.  **[P2: 检查权威数据库]**
        * 如果不是 `sibling`，调用 `get_caida_relationship` 查询它们的关系。
        * 这个工具返回一个字典。查看 `relationship` 键。
        * 如果 `relationship` 是 `peer` 或 `provider-customer`，则使用该结果。立即返回该结果 (并可附上 `direction` 信息)。

    3.  **[P3: 检查地点 (IXP 启发式)]**
        * 如果 CAIDA 数据库返回 `unknown`，则检查 `loc`。
        * `loc` 字段可能是一个 IXP 的名称。调用 `get_ixp_members_by_name(ixp_name=loc)`。
        * 这个工具返回一个字典。检查 `members` 列表。
        * 如果 AS A 和 AS B **都** 在 `members` 列表中，则关系 *很可能* 是 `peer`。
        * 在这种情况下，返回 `peer` (并注明是基于在 {loc} 的 IXP 成员身份推断的)。

    4.  **[P4: LLM 知识库]**
        * 如果以上所有步骤都返回 `unknown`（例如，CAIDA 未知，`loc` 也不是可识别的 IXP 或二者不在其中），此时才允许你使用你自己的（训练数据中的）储备知识进行“最佳猜测”。
        * 你的最终回答 **必须** 包含你得出结论的**完整推理链**。

    请开始工作。
    """

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt),
            ("human", "请分析: (AS A: {asn_a}, AS B: {asn_b}, Location: '{loc}')"),
            ("placeholder", "{agent_scratchpad}"), 
        ]
    )
    
    agent = create_openai_tools_agent(llm, tools, prompt)
    executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
    
    return executor

# ----------------------------------------------------------------------
#  3. 封装和运行
# ----------------------------------------------------------------------

def analyze_relationship(agent_executor: AgentExecutor, asn_a: int, asn_b: int, loc: str) -> str:
    """
    运行 Agent 分析并返回最终结果。
    """
    print(f"\n--- 正在分析: (AS A: {asn_a}, AS B: {asn_b}, Location: '{loc}') ---\n")
    
    input_data = {
        "asn_a": asn_a,
        "asn_b": asn_b,
        "loc": loc
    }
    
    try:
        response = agent_executor.invoke(input_data)
        print(f"\n--- 分析结束 ---\n")
        return response['output']
    except Exception as e:
        print(f"\n--- 分析出错: {e} ---\n")
        return f"分析过程中发生错误: {e}"

[CaidaLoader] 正在初始化...


In [ ]:
# --- 主程序入口 ---
os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxxxxxxxxxxxxxxxxxx"
if __name__ == "__main__":
    if "OPENAI_API_KEY" not in os.environ:
        print("错误: 请先设置 OPENAI_API_KEY 环境变量。")
    else:
        try:
            # 1. 在 Agent 启动前，必须先加载 CAIDA 数据
            caida_loader.load_data(LATEST_CAIDA_URL)

            # 2. 创建 Agent
            my_agent = create_as_relationship_agent()
            
            # 3. 运行不同场景的分析

            # 场景 1: Sibling 关系 (P1 命中)
            # AS15169 (Google) 和 AS32934 (Google)
            analyze_relationship(my_agent, 15169, 32934, "Google Cloud")
            
            # 场景 2: Provider-Customer (P2 命中)
            # 真实的 p2c 关系，例如 AS701 (Verizon) 是 AS703 (Verizon) 的 provider (内部关系)
            # 或者一个更经典的例子：AS3356 (Lumen) 和 AS209 (CenturyLink)
            # 我们来查一个已知的：AS3356 (Lumen/Level3) 和 AS2828 (XO Comm)
            analyze_relationship(my_agent, 3356, 2828, "Ashburn")

            # 场景 3: Peer (IXP 启发式, P3 命中)
            # 假设 AS2906 (Netflix) 和 AS3257 (T-Mobile) 在 CAIDA 中没有记录
            # 但它们都在 "DE-CIX Frankfurt"
            analyze_relationship(my_agent, 2906, 3257, "DE-CIX Frankfurt")

            # 场景 4: Peer (LLM 储备知识, P4 命中)
            # 假设两个 Tier-1 (例如 AS701 和 AS1239) 关系在 CAIDA 中未记录
            # 且地点是一个模糊的数据中心
            analyze_relationship(my_agent, 701, 1239, "Equinix Ashburn")

        except Exception as e:
            print(f"主程序运行失败: {e}")

错误: 请先设置 OPENAI_API_KEY 环境变量。
